**目次**<a id='toc0_'></a>    
- 1. [科学技術計算7課題](#toc1_)    
  - 1.1. [課題07-1：1次元のchord法の実装](#toc1_1_)    
  - 1.2. [課題07-2：多次元のchord法の実装](#toc1_2_)    
  - 1.3. [課題06-2：Broyden法の更新式の比較](#toc1_3_)    
  - 1.4. [課題07-3：SVDを用いたGN法](#toc1_4_)    
  - 1.5. [課題07-4：SVDを用いたLM法](#toc1_5_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[科学技術計算7課題](#toc0_)

**各課題で共通する注意事項**

- 関数にはアノテーション（引数や返り値の型情報）と`docstring`（関数の説明文）を必ず書く．

## 1.1. <a id='toc1_1_'></a>[課題07-1：1次元のchord法の実装](#toc0_)

演習資料では，ニュートン法を独自実装で，セカント法とHalley法はscipyが提供する関数を用いて1次元の非線形方程式の解を求めた．
それら以外にも，chord法と呼ばれる単純な方法がある．
これは，1次元のニュートン法の更新式において，
初期値$x_0$の微分$f'(x^{(0)})$を利用し続ける単純な方法である．

- 初期値$x^{(0)}$, $k=0$
- 収束するまで以下を反復
    1. $x^{(k+1)} = x^{(k)} - \frac{f(x^{(k)})}{f'(x^{(0)})}$
- 解$\hat{x} = x^{(k)}$を出力



**やること**

演習資料の`def newton_method_1d()`を参考にして，chord法を実装する．


**関数定義**

```python
def chord_1D(
        f: callable,
        g: callable,
        x0: float,
        maxiter: int,
        callback: callable,
) -> np.ndarray:
```

**実行手順**

- 目的関数は`Objective1D()`とする．
- `CacheXk1D`と`PlotCacheXk1D`を用いて収束をプロットし，ニュートン法の結果と比較せよ．
    - それぞれの初期値から最適解に収束しているかを確認すること
- 目的関数に対してアルゴリズムが収束するように，パラメータの値を調整すること
    - 初期値やパラメータによっては収束しない可能性があるので注意（発散する・収束途中，など）



## 1.2. <a id='toc1_2_'></a>[課題07-2：多次元のchord法の実装](#toc0_)

多次元（$n=m$）のニュートン法の更新式において，
初期値$\boldsymbol{x}^{(0)}$のヤコビ行列$J(\boldsymbol{x}^{(0)})$を利用し続ける単純な方法をchord法と呼ぶ．

chord法のアルゴリズムは以下のようになる．

- 初期値$\boldsymbol{x}^{(0)}$，$k=0$
- 収束するまで以下を反復
    - $\boldsymbol{d}^{(k)} = - J(\boldsymbol{x}^{(0)})^{-1} \boldsymbol{f}(\boldsymbol{x}^{(k)})$
        - $J(\boldsymbol{x}^{(0)}) \boldsymbol{d}^{(k)} = -  \boldsymbol{f}(\boldsymbol{x}^{(k)})$を解く
    - $\boldsymbol{x}^{(k+1)} = \boldsymbol{x}^{(k)} +\boldsymbol{d}^{(k)}$


**やること**

- 演習資料の`def newton_method()`を参考にしてchord法を実装する．
- 更新方向$\boldsymbol{d}^{(k)}$を求める連立方程式の係数行列$J(\boldsymbol{x}^{(0)})$は固定である．この事実を利用する．
  - 最初に$J(\boldsymbol{x}^{(0)})$を`scipy.linalg.lu_factor()`でLU分解する．
  - 毎回の反復における連立方程式は`scipy.linalg.lu_solve()`を用いて効率的に解く．


**関数定義**

```python
def chord(
        f: callable,
        J: callable,
        x0: np.ndarray,
        maxiter: int,
        callback: callable,
) -> np.ndarray:
```

**実行手順**

- 目的関数は`Function2D()`とする．
- `CacheXk`と`PlotCacheXk`を用いて収束をプロットし，ニュートン法の結果と比較せよ．
    - それぞれの初期値から最適解に収束しているかを確認すること
- 目的関数に対してアルゴリズムが収束するように，初期値やパラメータの値を調整すること
    - 初期値やパラメータによっては収束しない可能性があるので注意すること（発散する，振動して収束しない，など）

## 1.3. <a id='toc1_3_'></a>[課題06-2：Broyden法の更新式の比較](#toc0_)

演習資料では，ヤコビ行列の近似$B$を更新するBroyden法を紹介した．
Bryoden法には，他にもヘッセ行列の逆行列$H$を近似して更新する方法がある．

- ヤコビ行列 $B$ を近似して更新する場合
  - $B^{(k)} \boldsymbol{d}^{(k)} = -f(\boldsymbol{x}^{(k)})$を解いて$\boldsymbol{d}^{(k)}$を求める（つまり$\boldsymbol{d}^{(k)} = - (B^{(k)})^{-1} f(\boldsymbol{x}^{(k)})$）
- ヤコビ行列の逆行列 $H$ を近似して更新する場合
  - $\boldsymbol{d}^{(k)} = - H^{(k)} f(\boldsymbol{x}^{(k)})$

近似ヤコビ逆行列$H$の更新式には，good Broyden methodとbad Broyden methodの2つが知られている．この課題ではこれらの実装と比較を行う．

- 参考：https://en.wikipedia.org/wiki/Broyden%27s_method

**やること**

- good/bad Broyden methodの更新式を調べて，合計3つの更新式を関数として実装する．
  - 更新式は類似しているため，同じ関数を，呼び出す引数を変えるだけですむ実装にすること．
- 演習資料の関数`def broyden_method()` を参考にして，準ニュートン法を実装する．
    - 引数 `update` で「ヤコビ行列$B$を更新する」か「逆行列$H$を更新する」かを切り替えられるようにする．
    - `update="H"`の場合には，引数 `method` で更新式（"good Broyden", "bad Broyden"）を選べるようにする．

**関数定義**

```python
def broyden_method(
        f: callable,
        x0: np.ndarray,
        B0: np.ndarray,
        update: str,  # "B" or "H"
        method: str,  # "good Broyden", "bad Broyden"
        maxiter: int,
        callback: callable,
) -> np.ndarray:
```

**実行手順**

- 目的関数は`Function2D()`とする．
- 3通りの更新方法（B, good H, bad H）について，`CacheXk`を用いてそれぞれ解系列を記録し，`PlotCacheXk`を用いて1つの図にプロットする．
  - 収束の速さや軌跡を比較し，差異を観察する（収束の速さや安定性に違いがあるか，初期値依存性が強い更新式はどれか，などについて）．
- 目的関数に対してアルゴリズムが収束するように，初期値やパラメータの値を調整すること
    - 初期値やパラメータによっては収束しない可能性があるので注意すること（発散する，振動して収束しない，など）
- 初期値を変えて，同様に比較する．


## 1.4. <a id='toc1_4_'></a>[課題07-3：SVDを用いたGN法](#toc0_)


演習資料の`gauss_newton_pinv()`では，
$$J \boldsymbol{d} = -  \boldsymbol{f}$$
を解くために，擬似逆行列を用いて（実際には`np.linalg.lstsq()`を利用して）更新方向を
$$\boldsymbol{d} = - J^+ \boldsymbol{f}$$
と求めるGN法を実装した．

この課題では，SVDを用いて（擬似）逆行列を計算するGN法を実装する．

**やること**

- 演習資料の`def gauss_newton_pinv()`を参考にして，SVDを用いて（擬似）逆行列を計算するGN法`gauss_newton_svd()`を実装する．
  - ヤコビ行列を$J  = U W V^T$とSVDし，これを用いて（疑似）逆行列$J^+$を構成する．1ステップ分のアルゴリズムは以下のようになる．
    - SVDを計算する：$J(\boldsymbol{x}^{(k)})  = U W V^T$
    - $\boldsymbol{d}^{(k)} = - V W^{+} U^T \boldsymbol{f}(\boldsymbol{x}^{(k)})$
    - $\boldsymbol{x}^{(k+1)} = \boldsymbol{x}^{(k)} +\boldsymbol{d}^{(k)}$

ここで$W^{+}$は，ランク落ちしている場合の擬似逆行列を構成する対角行列である（$J$のSVDで得られた特異値からなる対角行列$W$の，しきい値以下の対角成分を0にし，しきい値を上回る対角成分を逆数にしたもの）．



**関数定義**

```python
def gauss_newton_svd(
        f: callable,
        J: callable,
        x0: np.ndarray,
        alpha: float,
        maxiter: int,
        callback: callable,
) -> np.ndarray:
```

**実行手順**

- 目的関数は`Function2D(m=3)`とする．
- SVDの計算には`np.linalg.svd()`または`scipy.linalg.svd()`を用いること．
- `CacheXk`と`PlotCacheXk`を用いて収束をプロットし，通常のGN法の結果と比較せよ（同一の結果であることを確認すること）．
    - 複数の初期値から最適解に収束しているかを確認すること


## 1.5. <a id='toc1_5_'></a>[課題07-4：SVDを用いたLM法](#toc0_)


演習資料の`levenberg_marquardt()`では，
$$(J^T J + \lambda \boldsymbol{I}) \boldsymbol{d} = - J^T \boldsymbol{f}$$
を解いて更新方向で求めた．
ここでは$\boldsymbol{I}$は単位行列$I$であるとすると，更新方向は
$$\boldsymbol{d} = - (J^T J + \lambda I)^{-1} J^T \boldsymbol{f}$$
となる．

この課題では，SVDを用いて（擬似）逆行列を計算するLM法を実装する．

**やること**

ヤコビ行列のSVDを$J = U W V^T$とすると，
LM法の右辺の行列部分は以下のように展開できる．

\begin{align*}
(J^T J + \lambda I)^{-1} J^T
&=
((U W V^T)^T (U W V^T)  + \lambda I)^{-1} (U W V^T)^T
\\
&=
(V W^2 V^T + \lambda I)^{-1} V W U^T
\\
&=
(V \mathrm{diag}(\sigma_1^2 + \lambda, \ldots, \sigma_p^2 + \lambda) V^T)^{-1} V W U^T
\\
&=
V \mathrm{diag}(1/(\sigma_1^2 + \lambda), \ldots, 1/(\sigma_p^2 + \lambda)) V^T V W U^T
\\
&=
V \mathrm{diag}(1/(\sigma_1 + \lambda/\sigma_1), \ldots, 1/(\sigma_p + \lambda/\sigma_p)) U^T
\\
&=
V W^+_\lambda U^T
\end{align*}
ここで$W^+_\lambda$は最後から2行目の，対角行列の部分を表す．


これを用いて，SVDを用いて（擬似）逆行列を計算するLN法`levenberg_marquardt_svd()`を実装する．

- 演習資料の`def levenberg_marquardt()`を参考にして，SVDによる擬似逆行列を用いたLN法`levenberg_marquardt_svd()`を実装する．
  - ヤコビ行列を$J  = U W V^T$とSVDし，これを用いて（疑似）逆行列を構成する．1ステップ分のアルゴリズムは以下のようになる．
    - SVDを計算する：$J(\boldsymbol{x}^{(k)})  = U W V^T$
    - $\boldsymbol{d}^{(k)} = - V W^+_{\lambda^{(k)}} U^T \boldsymbol{f}$
    - 以下は通常のLN法と同様

**関数定義**

```python
def levenberg_marquardt_svd(
        f: callable,
        J: callable,
        x0: np.ndarray,
        lmd: float,
        alpha: float,
        gamma1: float,
        gamma2: float,
        maxiter: int,
        callback: callable,
) -> np.ndarray:
```


**実行手順**

- 目的関数は`Function2D(m=3)`とする．
- SVDの計算には`np.linalg.svd()`または`scipy.linalg.svd()`を用いること．
- `CacheXk`と`PlotCacheXk`を用いて収束をプロットし，通常のLN法の結果と比較せよ（同一の結果であることを確認すること）．
    - 複数の初期値から最適解に収束しているかを確認すること
